# LVLMs-Saliency — Colab Setup
Run each cell in order. Requires a GPU runtime (T4 or better).
Runtime → Change runtime type → GPU

In [ ]:
# Cell 1: Check GPU
import torch
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# Cell 2: Clone repo and install dependencies
!git clone https://github.com/qiiizhen/LVLMs-Saliency.git
%cd LVLMs-Saliency

# Install LLaVA from local source
!pip install -e src/LLaVA/ -q

# Install other dependencies
!pip install transformers==4.37.2 accelerate bitsandbytes datasets gradio Pillow seaborn tqdm -q

print('Done.')

In [ ]:
# Cell 3: Download LLaVA-1.5-7B model
# This takes ~10-15 minutes. The model is 13GB.
from huggingface_hub import snapshot_download
snapshot_download(
    repo_id='liuhaotian/llava-v1.5-7b',
    local_dir='/content/llava-v1.5-7b'
)
print('Model downloaded.')

In [ ]:
# Cell 4: Upload 499775_hall.pt
# This file is the step1 output (only 1.8KB).
# It should already be in the cloned repo.
import os
if os.path.exists('499775_hall.pt'):
    print('499775_hall.pt found — ready to run step2.')
else:
    print('Not found — upload it manually:')
    from google.colab import files
    files.upload()  # upload 499775_hall.pt from your computer

In [ ]:
# Cell 5: Enable 4-bit quantization (required for T4 16GB)
# Patch InferenceArgs in demo_step2_llava.py to use load_4bit=True
with open('demo_step2_llava.py', 'r') as f:
    code = f.read()

code = code.replace('load_4bit = False', 'load_4bit = True')

with open('demo_step2_llava.py', 'w') as f:
    f.write(code)

print('4-bit quantization enabled.')

In [ ]:
# Cell 6: Run step2
import sys
sys.path.insert(0, 'src/transformers/src')  # custom tokenizer (must be first)
sys.path.insert(1, 'src/LLaVA')

!python demo_step2_llava.py \
    --model-path /content/llava-v1.5-7b

In [ ]:
# Cell 7: View output saliency map
from IPython.display import Image as IPImage, display
import glob

for f in sorted(glob.glob('onlytext_hall_499775*.png')):
    print(f)
    display(IPImage(f))

In [ ]:
# Cell 8: Run quantitative analysis
import sys
sys.path.insert(0, 'src/transformers/src')
sys.path.insert(1, 'src/LLaVA')

!python analyze_from_png.py